# Forecasting sales<a id='top'></a>

### Contents

* [EDA](#eda)
* [Feature engineering](#eng)
* [Model training](#models)
* [Evaluation](#eval)

In [ ]:
from time import time
t_start = time()
t_end = t_start + 12*60*60

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import re
import functools as ft

import gc
import pickle
from os.path import exists
from statsmodels.tsa.deterministic import DeterministicProcess
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from statsmodels.graphics.tsaplots import plot_pacf
from statsmodels.tsa.stattools import pacf
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, RandomizedSearchCV
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_log_error as msle
from sklearn.multioutput import MultiOutputRegressor
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from scipy.stats import beta, uniform, nbinom
from sklearn.impute import SimpleImputer


def reset():
    global df
    df = {}
    import os
    for dirname, _, filenames in os.walk('../input/store-sales-time-series-forecasting'):
        for filename in filenames:
            file = os.path.join(dirname, filename)
            description = filename.replace('.csv','')
            df[description] = pd.read_csv(file)

In [ ]:
# Basic preprocessing

reset()  # import data

# Set the index for each df

def choose_index(table, column, date=False, date_col=None):
    '''Takes table and column (or list of columns) and sets it to index. 
    If it is dattime then it converts it to a period index.
    If a list is passed and date is True, then a date column needs to be provided.
    '''
    def set_to_period(series):
        series = pd.to_datetime(series)
        return series #.dt.to_period('D')  
    
    if date and isinstance(column,str):
        df[table][column] = set_to_period(df[table][column]) # If single-level index, it is set to period type
    elif date and isinstance(column,list):
        if date_col in column: # if multi-level index, find date column and set that to period type
            df[table][date_col] = set_to_period(df[table][date_col])
    elif date:
        raise TypeError('For hierrachical index with date, the date column is required as date_col argument')

    
    df[table].set_index(column, inplace=True) # establish index
    
    
choose_index('oil', 'date', date=True)
choose_index('sample_submission', 'id')
choose_index('holidays_events', 'date', date=True)
choose_index('stores', 'store_nbr')
for table in ['train','test']:
    choose_index(table, ['id','date'],date=True,date_col='date')
choose_index('transactions', 'date', date=True)


families = list(df['train']['family'].unique())

In [ ]:
# Set int types
df['train']['store_nbr'] = df['train']['store_nbr'].astype('int')
df['train']['family'] = df['train']['family'].astype('category')
df['test']['store_nbr'] = df['test']['store_nbr'].astype('int')
df['test']['family'] = df['test']['family'].astype('category')

# EDA<a id='eda'></a>

* [Missing values in the raw tables](#missing)
* [Data types](#datatypes)
* [Basic visualisations](#basicviz)
* [Seasonality](#seasonaleda)
* Inspect impact of potential leading indicators:
    * [oil](#oil)
    * [holidays and events](#holidays)
    * [promotions](#promo)
    
[Return to top](#top)

## Missing values in the raw tables<a id='missing'></a>

In the raw tables, the only missing data is within the oil table. We use linear interpolation for the dates other than for the first value in the table (2013-01-01) which is missing. We will just pull back the value from the following day for this value.

In [ ]:
# Count the missing values for each table and column.

for table in df.keys():
    print(table)
    display(df[table].isna().sum())
    print('-------------------')

In [ ]:
# Fill the missing values in the oil table

df['oil']['dcoilwtico'] = df['oil']['dcoilwtico'].interpolate('linear')
df['oil'].iloc[0,0] = df['oil'].iloc[1,0]

## Datatypes<a id='datatypes'></a>

The only datatypes we need to fix are identifying and assigning features to have a categorical datatype.

In [ ]:
# Change columns to category datatype

cat_cols = {}
cat_cols['holidays_events'] = ['type', 'locale', 'locale_name']
cat_cols['stores'] = list(df['stores'].columns)
cat_cols['train'] = ['store_nbr', 'family']
cat_cols['test'] = ['store_nbr', 'family']
cat_cols['transactions'] = ['store_nbr']

for table, col in cat_cols.items():
    df[table][col].astype('category')

## Basic visualisations<a id='basicviz'></a>

We focus on the training set, inspecting the different product familes. We first see which have the largest numbers, but take this with a pinch of salt since we don't know about the units in which the records are taken.

Upon closer inspection of the top three product families, we notice that produce, the third highest, has some worrying periods of zero (or close to zero) sales recorded, suggesting some sort of logging error. There are also issues with the beverages amounts for the same periods, thought their numbers are lower but not zero.

In [ ]:
_,_ = plt.subplots(figsize=(8,12))
sns.barplot(data= df['train'].groupby('family')['sales'].agg(['mean']).sort_values(by = 'mean',ascending=False).reset_index(),
            x='mean',
            y='family')
plt.title('The product families with the largest sale numbers')
plt.show()

In [ ]:
cols_to_show = df['train'].groupby('family')['sales'].agg(['mean']).sort_values(by = 'mean',ascending=False).reset_index().iloc[0:3,0].to_list()

store = 5
plot = {}
fig,ax = plt.subplots(figsize=(25,12))
for col in cols_to_show:
    plot[col] = df['train'][(df['train'].store_nbr == store) & (df['train'].family == col)]['sales'].reset_index().plot(ax=ax,x='date',
                                                                                                                        y='sales',
                                                                                                                        label=col,
                                                                                                                        lw=0.6)
ax2 = ax.twinx()
plot['transactions'] = df['transactions'][df['transactions'].store_nbr == store].reset_index().plot(ax=ax2,
                                                                                                    x='date',
                                                                                                    y='transactions',
                                                                                                    color = 'red',
                                                                                                    lw=0.7)
ax2.legend(loc='lower right')
plt.title(f'Sales of the top {len(cols_to_show)} product families and total number of transactions at store {store}')
plt.show()

Looking at the produce family, overall across all stores there appears to be some error with regard to logging during periods in 2014 and 2015. The overall average sales shows sharp drops for the same periods, so it appears the sales are missing in the system or misrecorded (as opposed to being mislabelled).

Further inspection reveals that multiple families have issues regarding logs over dates in 2013-2015. There is a list

In [ ]:
temp_df = df['train'].groupby(['date','family'])['sales'].mean().reset_index()
problem_families = []

for family in families:
    a = temp_df[(temp_df['date'] == '2015-01-15') & (temp_df['family'] == family)]['sales'].to_list()
    b = temp_df[(temp_df['date'] == '2015-11-15') & (temp_df['family'] == family)]['sales'].to_list()
    a1 = temp_df[(temp_df['date'] == '2014-04-15') & (temp_df['family'] == family)]['sales'].to_list()
    if a[0] * 3 < b[0] or a1[0]*3 < b[0]:
        problem_families.append(family)
        
ok_families = ['AUTOMOTIVE', 'BEAUTY', 'HARDWARE', 'SCHOOL AND OFFICE SUPPLIES']
for family in ok_families:
    problem_families.remove(family)
    
print(f'The problem families are: {problem_families}')


del temp_df

In [ ]:
temp_df = df['train'][df['train']['family'] == 'PRODUCE'].groupby('date')['sales'].agg(['mean','min','max'])
fig,ax = plt.subplots(figsize=(25,10))
for col in temp_df.columns:
    temp_df.reset_index().plot(x='date',y=col, ax=ax, lw=0.8)
plt.title('Average, min and max sales of produce across all stores')
plt.show()

temp_df = df['train'].groupby('date')['sales'].agg(['mean'])
fig,ax = plt.subplots(figsize=(25,8))
for col in temp_df.columns:
    temp_df.reset_index().plot(x='date',y=col, ax=ax, lw=0.8)
plt.title('Average, min and max sales across all stores and product families')
plt.show()

del temp_df

In [ ]:
temp_df = df['train'][df['train']['family'] == 'PRODUCE'][['store_nbr','sales']].reset_index()#.drop('id',axis=1,inplace=True)

temp_df.drop('id',axis=1,inplace=True)

temp_df1 = df['train'][df['train']['family'] == 'GROCERY I'][['store_nbr','sales']].reset_index()

temp_df1.drop('id',axis=1,inplace=True)

temp_df = temp_df.merge(temp_df1,on=['date','store_nbr'])

temp_df['ratio'] = temp_df['sales_x']/temp_df['sales_y']

store=23

_,ax = plt.subplots(figsize=(25,7))
temp_df[temp_df['store_nbr']==store].plot(kind='line',x='date', y='ratio', ax=ax, lw=0.5, color='red', label = 'ratio')
ax.legend(loc='lower right')
ax2=ax.twinx()
temp_df[temp_df['store_nbr']==store].plot(kind='line',x='date', y='sales_x',  lw=0.5, ax=ax2, label='Produce')
temp_df[temp_df['store_nbr']==store].plot(kind='line',x='date', y='sales_y',  lw=0.5, ax=ax2, label='Grocery I')
plt.show()

del temp_df
del temp_df1

In [ ]:
temp_df = df['train'].groupby(['date','family'])['sales'].mean().reset_index()

fig,ax=plt.subplots(3,4,figsize=(20,12))
for n, family in enumerate(problem_families):
    temp_df[temp_df['family']==family].plot(x='date', y='sales', ax=ax[n//4,n%4])
    ax[n//4,n%4].set_title(family)
    plt.subplots_adjust(wspace=0.15,hspace=0.4)
plt.show()

del temp_df

## Exploring seaonal dependence<a id='seasonaleda'></a>

First we explore whether the month is an indicator. We see that December has, as expected, a substantial bump in sales each year, while November perhaps has a slight increase. Otherwise there are no clear trends. (This is not helped by the apparent errors in the 2013-2015 data, causing for example the fluctuations in the 2014 line).

As for the day of the week, there is a clear increase in sales over the weekend, with most sales made on Sundays. During the week, there is a dip on Thrusdays, and another smaller dip on Tuesdays.

We can also see the effect of the bimonthly public sector paychecks, with more sales earlier in the month, and a smallbump in the middle.

To conclude, it is important to know:

* if the month is December or not,
* which day of the week it is,
* how long since public sector workers received their last paycheck.

In [ ]:
temp_df = df['train'].groupby('date')['sales'].mean().reset_index()

# For month-by-month behaviour
temp_df1= temp_df.resample('M',on='date').mean()
temp_df1['Month'] = temp_df1.index.month
temp_df1['Year'] = temp_df1.index.year

# For day-of-the-week behaviour
temp_df2=temp_df.copy()
temp_df2['Day'] = temp_df2['date'].dt.day_of_week
temp_df2['Week'] = temp_df2['date'].dt.isocalendar().week
temp_df2['Year'] = temp_df2['date'].dt.isocalendar().year

# For day-of-week averags
aggs=['mean','median','min','max']
temp_df3 = temp_df2.groupby('Day')['sales'].agg(aggs).reset_index()

# For day-of-month
temp_df4 = temp_df.copy()
temp_df4['Day'] = temp_df4['date'].dt.day
temp_df4 = temp_df4.groupby('Day')['sales'].agg(aggs).reset_index()

# Month plot
years = list(temp_df1['Year'].unique())
fig,ax=plt.subplots(figsize=(16,6))
for year in years:
    temp_df1[temp_df1['Year'] == year].plot(x='Month', y='sales',label=year, ax=ax)
plt.title('Average sales for each month')
plt.show()

# Day-of-the-week plot 1
weeks = list(temp_df2['Week'].unique())
fig, ax = plt.subplots(figsize=(16,6))
for year in years:
    for week in weeks:
        temp_df2[(temp_df2['Week']==week) & (temp_df2['Year'] == year)].plot(x='Day', y='sales', ax=ax, legend=None)
plt.title('Average sales by day of the week')
plt.show()

# Day of week averages plot
fig,ax=plt.subplots(figsize=(16,6))
for agg in aggs:
    temp_df3.plot(x='Day', y=agg, ax=ax)
plt.title('Average weekly trends')
plt.show()

# Paycheck cycles
fig,ax=plt.subplots(figsize=(16,6))
for agg in aggs:
    temp_df4.plot(x='Day', y=agg, ax=ax)
plt.title('Average sales on day-of-month')
plt.show()

del temp_df
del temp_df2
del temp_df3


### School seasons

The sales of SCHOOL AND OFFICE SUPPLIES is clearly heavily dependent upon the school season. We see below that the sales peak in april, may, august and september.

In [ ]:
temp_df= df['train'][df['train']['family'] == 'SCHOOL AND OFFICE SUPPLIES'].reset_index().groupby('date')[['sales','onpromotion']].agg('mean')

fig,ax = plt.subplots(figsize=(14,6))
temp_df[temp_df.index.year == 2016].plot(ax=ax)
plt.xticks(ticks = [pd.Timestamp(f'2016-{n}-01') for n in range(1,13)])
plt.title('School and office supply average sales in 2016')
plt.show()

temp_df=None

## Exploring for leading factors

### Oil<a id='oil'></a>

The graphics below show that while the overall number of transactions did not decrease after the oil price dropped, the average number of sales (i.e. how much is purchased) did.

In [ ]:
# First plot oil price vs time
fig,ax = plt.subplots(1,3,figsize=(25,6))
df['oil'].plot(ax=ax[0], legend=None)
ax[0].set_title('Oil price over time')
ax[0].set_xlabel('Date')
ax[0].set_ylabel('Oil price')


# First create a DF with oil and transactions
temp_df = pd.merge(left = df['transactions'], right = df['oil'], how = 'left', left_index=True, right_index=True).reset_index().groupby('date')[['transactions', 'dcoilwtico']].mean()

low=temp_df['transactions'].quantile(0.1)
high=temp_df['transactions'].quantile(0.9)

# Plot

sns.regplot(data = temp_df[(temp_df['transactions']<high) & (temp_df['transactions']>low)],
               x='dcoilwtico',
               y='transactions',
           ax=ax[1])
ax[1].set_title(f'Comparing oil price with average number of transactions')
ax[1].set_xlabel('Oil price')
ax[1].set_ylabel('Number of transactions')


# Next create a DF with sales and transactions
temp_df = pd.merge(left = df['train'], right = df['oil'], how = 'left', left_index=True, right_index=True).reset_index().groupby('date')[['sales', 'dcoilwtico']].mean()

temp_df
low=temp_df['sales'].quantile(0.1)
high=temp_df['sales'].quantile(0.9)

# Plot
sns.regplot(data = temp_df[(temp_df['sales']<high) & (temp_df['sales']>low)],
               x='dcoilwtico',
               y='sales',
               ax=ax[2])
plt.title(f'Comparing oil price with average sales')
plt.xlabel('Oil price')
plt.ylabel('Average sales')
plt.show()

temp_df=None

## Holidays<a id='holidays'></a>

We see in the plot below for 2016 that holidays and other events do have an impact on the sales. Their impact is not consistent though and appears to depend on the occasion. For example:

* There is a rising trend in the build-up to Christmas, and then New Years day has close to zero sales (presumably most stores are closed).
* Sales are remain on the higher end after the Batella de Pichincha, while after the Primer Grito de Independencia the sales remain around their average (rather than fluctuating up and down).
* The earthquake caused a surge in sales.
* Alcohol salese see a surge around cerain holdays, with Christmas being the most substantial.

In [ ]:
# Set up temporary DFs
temp_df=df['train'].groupby('date')['sales'].mean().reset_index()
temp_df['rolling_7'] = temp_df['sales'].rolling(7).mean()
temp_df['rolling_16'] = temp_df['sales'].rolling(16).mean()
temp_df1 = df['holidays_events'][(df['holidays_events']['locale'] == 'National') & (df['holidays_events']['transferred'] == False)].reset_index()
temp_df2 = df['train'][df['train']['family'] == 'LIQUOR,WINE,BEER'].groupby('date')['sales'].mean().rolling(window=7).agg(['min','max']).reset_index()

# Plot 2016
fig,ax=plt.subplots(figsize=(25,8))
temp_df[temp_df['date'].dt.year == 2016].plot(x='date', y='sales', ax=ax, lw=0.8, legend='daily sales')
temp_df[temp_df['date'].dt.year == 2016].plot(x='date', y='rolling_7', ax=ax, lw=0.8, legend='7-day rolling average')
plt.vlines(x=temp_df1[temp_df1['date'].dt.year == 2016].date, color='red', ymin=0,ymax=800, lw=0.5, alpha=0.7)
plt.title('Average sales in 2016, showing holidays')
plt.ylabel('7-day rolling average of sales')
labels = [94,95,97,98,123,132,133,134,135,145]
for n in labels:
    if n in [123,134]:
        delta = -40
    else:
        delta=0
    plt.annotate(temp_df1.loc[n,'description'], (temp_df1.loc[n,'date'], 805+delta))
    plt.plot(temp_df1.loc[n,'date'], 795, 'go')
plt.legend(loc='lower right')
ax2=ax.twinx()
ax2.plot(temp_df2[temp_df2['date'].dt.year == 2016]['date'],
         temp_df2[temp_df2['date'].dt.year == 2016]['min'],
         label = 'alcohol 7-day average min', c = 'black')
ax2.plot(temp_df2[temp_df2['date'].dt.year == 2016]['date'],
         temp_df2[temp_df2['date'].dt.year == 2016]['max'],
         label = 'alcohol 7-day average max', c = 'black')
plt.legend(loc = 'upper right')
plt.show()

# Plot all

types = ['Holiday', 'Transfer', 'Additional', 'Bridge', 'Work Day', 'Event']
colors = ['red','red','red','red','blue','green']
markers = ['o','>','+','P','^','X']
fig,ax=plt.subplots(figsize=(25,8))
temp_df.plot(x='date', y='rolling_16', ax=ax, lw=0.8, legend=None)
for t,c,m in zip(types, colors, markers):
    plt.vlines(x=temp_df1[temp_df1['type'] == t].date, color=c, ymin=0, ymax=800, lw=0.5, alpha = 0.5)
    plt.scatter(temp_df1[temp_df1['type'] == t].date,[795]*len(temp_df1[temp_df1['type'] == t].date),c=c, marker=m, label=t)
plt.title('Average sales, showing holidays')
plt.ylabel('16-day rolling average of sales')
plt.legend()
plt.show()

temp_df=None
temp_df1=None

## Promotions<a id='promo'></a>

We see in the scatter graph below the unsurprising connecting between promotions and sales. If there are more items on promotion, there are more sales.

In [ ]:
temp_df = df['train'].groupby('date')[['sales','onpromotion']].mean()

_,ax = plt.subplots(figsize=(10,8))
sns.regplot(data=temp_df, y='sales', x='onpromotion', ax=ax)
plt.title('Impact of promotions on sales, averaged across all stores and products')
plt.ylabel('Average sales')
plt.xlabel('Number of promotions')
plt.show()

temp_df=None

# Feature engineering<a id='eng'></a>

* [Encoding of categorical data](#ohe)
* [Time-step feature](#timestep)
* [Lag features](#lag)
* [Seasonal indicators](#seasonal):
    * Day of the week
    * December
    * Days since last paycheck
    * New year's day
    * School year
* [Leading indicators](#leading) from EDA:
    * oil
    * holidays
    * promotions
    
 Functions are defined at each step, before [applying the functions](#create) at the end. There is the option to either create the data or load it from a saved file.
    
[Return to top](#top)

## Encoding the categories<a id='ohe'></a>

Rather than a traditional one hot encoding method we are going to create a dataframe for each product family, and each store. These will be stored in a dictionary and saved to a pickle file.

In this step we are also creating time-step features for each table. The time-step will start 2013-01-01 unless that store opened at a later date, in which case that date is used.

Some stores also do not sell allproduct families. When this is observed, a predictions table with all zeros is created and the corresponding training table is deleted.

In [ ]:
def save_to_pickle(obj, filename):
    with open(filename, 'wb') as file:
        pickle.dump(obj, file, protocol=-1)

stores=list(df['train']['store_nbr'].unique())

def load_train(file):
    global train
    global train_loaded
    if exists(file):
        with open(file, 'rb') as file1:
            train = pickle.load(file1)
        print('Loaded training data from saved file.')
        train_loaded=True
    else:
        print('Training data not found')
        train_loaded=False

def load_pred(file):
    global predictions
    global preds_loaded
    if exists(file):
        with open(file, 'rb') as file1:
            predictions = pickle.load(file1)
        print('Loaded predictions from saved file.')
        preds_loaded = True
    else:
        print('Predictions data not found')
        preds_loaded = False

def initialize_dfs():
    # Create separate dataframes for each family and store.
    global train
    global predictions
    train = {}
    predictions = {}
    for family in families:
        train[family] = {}
        predictions[family]= {}
        t0=time()
        for store in stores:
            do_time_step = True
            train[family][store] = df['train'][(df['train']['family'] == family) & (df['train']['store_nbr'] == store)].drop(['store_nbr','family'],axis=1).reset_index().drop('id',axis=1).set_index('date')
            predictions[family][store] = df['test'][(df['test']['family'] == family) & (df['test']['store_nbr'] == store)].drop(['store_nbr','family'],axis=1).reset_index().drop('id',axis=1).set_index('date')
            # Delete leading zeros if store opened after 2013-01-01
            if train[family][store].loc[pd.Timestamp('2013-01-02'),'sales'] == 0: # checks Jan 2 2013 for zero sales
                if (train[family][store]['sales'] > 0).sum() > 0: # if some sales for that store and family are non-zero
                    open_date = train[family][store][train[family][store]['sales'] > 0].index[0] # find date store was opened
                    train[family][store] = train[family][store].loc[open_date:] # catch only data from that date onwards
                else: # if no sales for that family at that store, then create a prediction table with all zeros and delete the corresponding training df
                    predictions[family][store]['sales'] = [0]*16
                    del train[family][store]
                    do_time_step = False
            
            if do_time_step:
                # Create time-step feature
                dp = DeterministicProcess(index = train[family][store].index,
                                              order = 1)
                time_steps = dp.in_sample()
                train[family][store] = pd.merge(left = train[family][store],
                                                right = time_steps,
                                                left_index = True,
                                                right_index = True)
                train[family][store].rename(columns = {'trend':'time_step'}, inplace=True)

                predictions[family][store]['time_step'] = dp.out_of_sample(steps=16, forecast_index = pd.date_range(start="2017-08-16",
                                                                                                       end="2017-08-31"))['trend']
        print(f'DF created for {family} in {time()-t0:.2f} seconds')

## Lag features<a id='lag'></a>

We create lag features. For each store and product family we inspect the partial autocorrelation. The lags with sufficient correlation are then chosen, but only those that ar emore than 16 days (the length of the forecast needed) and lag features created (the lags will depend on the store and product family). The initial values for each lag feature are filled with the median values.

In [ ]:
def create_lags(nlags):
    for family in families:
        t0=time()
        for store in train[family].keys():
            if not 'sales_t-7' in train[family][store].columns:
                lags = [ n for n,x in enumerate(list(pacf(train[family][store]['sales'],
                                                          nlags=min(nlags, len(train[family][store])//2-1),
                                                          method='ywm'))) 
                        if np.abs(x)>0.1 and n>0
                       ]
                for n in lags:
                    if n > 16:
                        train[family][store][f'sales_t-{n}'] = train[family][store]['sales'].shift(n)
                        train[family][store][f'sales_t-{n}'].fillna(np.nanmedian(train[family][store][f'sales_t-{n}'].values),inplace=True)
                        predictions[family][store][f'sales_t-{n}'] = list(train[family][store].iloc[-1-n:15-n]['sales'].copy())
        print(f'Lag features done for {family} in {time()-t0:.2f} sec')

## Seasonal indicators<a id='seasonal'></a>

Following our [analysis above](#seasonaleda), we will add features that indicate the day of the week and whether it is December or not.

We also add a feature counting the days since the last paycheck was received for public workers. They are received on the 15th and last day of each month. 

Then we add an indicator for New Year's Day, when most stores appear to be closed.

Finally we add indicators for the two school seasons.

In [ ]:
def seasonal_inds():
    # Create indicator dataframe for day of the week
    dates = pd.DataFrame(index=pd.date_range('2013-01-01', '2017-12-31'))
    dates['Day'] = dates.index.day_of_week
    dates = pd.DataFrame(OneHotEncoder(drop='first',sparse=False).fit_transform(dates[['Day']]),
                             columns = ['Tues','Weds','Thurs','Fri','Sat','Sun'], 
                             index = dates.index)

    # Add column for December
    dates['December'] = dates.index.month
    dates['December'] = dates['December'].apply(lambda x : 1 if x == 12 else 0)
    dates['December'] = dates['December'].astype('int')

    # Add column for days since paycheck
    dates['Days since paycheck'] = dates.index.day
    dates['adjust'] = dates.index.daysinmonth - dates.index.day
    dates['adjust'] = (dates.index.daysinmonth - 15) * dates['adjust'].apply(lambda x : 1 if x == 0 else 0)
    dates['Days since paycheck'] = dates['Days since paycheck'].apply(lambda x : x if x < 15 else x-15) - dates['adjust']
    dates.drop('adjust', axis=1, inplace=True)

    # Add column for new year's day
    dates['NYD'] = dates.index.dayofyear
    dates['NYD'] = dates['NYD'].apply(lambda x : 1 if x == 1 else 0)
    dates['NYD'] = dates['NYD'].astype('int')
    
    # Add column for school year
    monthday = pd.Series(dates.index.month)*100 + pd.Series(dates.index.day)
    SY1 = monthday.apply(lambda x : 1 if x<=515 and x >= 415 else 0)
    SY2 = monthday.apply(lambda x : 1 if x<=913 and x >= 812 else 0)
    dates['School year 1'] = list(SY1)
    dates['School year 2'] = list(SY2)
    dates['School year 1'] = dates['School year 1'].astype('int')
    dates['School year 2'] = dates['School year 2'].astype('int')
    
    
    for family in families:
        
        if family == 'SCHOOL AND OFFICE SUPPLIES':
            cols = list(dates.columns)
        else:
            cols = list(dates.columns)
            cols.remove('School year 2')
            cols.remove('School year 1')
        
        # Join the dates dataframe to each training table
        for store in train[family].keys():
            train[family][store] = pd.merge(left=train[family][store], 
                                             right=dates[cols],
                                             left_index=True,
                                             right_index=True,
                                             how='left'
                                           )
            predictions[family][store] = pd.merge(left=predictions[family][store],
                                                  right=dates[cols],
                                                  left_index=True,
                                                  right_index=True,
                                                  how='left'
                                                 )

## Leading indicators<a id='leading'></a>

### Holidays and events

For now, we will ignore holidays and events. We know Christmas has a significant effect on sales, but this is (at least partly) accounted for in the seasonal indicators. The other holidays do not appear to have a substantial effect.

(For later: The holidays table lists the locale type (regional or local). If it is regional then the locale_name is the state; if it is local then local_name is the city. We can use these to connect holidays to stores.)

In [ ]:
def holiday_sales():
    for family in families:
        for store in train[family].keys():
            holiday_mask = (df['holidays_events']['locale_name'] == df['stores'].loc[store]['city'])\
            | (df['holidays_events']['locale_name'] == df['stores'].loc[store]['state'])\
            | (df['holidays_events']['locale'] == 'National')

            temp_df1 = df['holidays_events'][holiday_mask].reset_index()
            temp_df = train[family][store]['sales'].rolling(window=7, center=True).agg(['min','max'])
            temp_df['lag_max-7'] = temp_df['max'].shift(7)
            temp_df['lag_min-7'] = temp_df['min'].shift(7)
            temp_df['Jump from 7'] = temp_df['max'] / temp_df['lag_max-7']

    #         fig,ax = plt.subplots(figsize=(30,10))
    #         temp_df.plot(y=['min','max'], ax=ax, c='black')
    #         for t,c,m in zip(types, colors, markers):
    #             plt.vlines(x=temp_df1[temp_df1['type'] == t].date, color=c, ymin=0, ymax=600, lw=0.5, alpha = 0.5)
    #             plt.scatter(temp_df1[temp_df1['type'] == t].date,[600]*len(temp_df1[temp_df1['type'] == t].date),c=c, marker=m, label=t)
    #         ax2 = ax.twinx()
    #         ax2.plot(temp_df.index, temp_df['Jump from 7'], lw=0.7)
    #         plt.show()


            temp_df2 = pd.merge(left = temp_df1,
                                right = temp_df,
                                left_on = 'date',
                                right_index = True,
                                how = 'left'
                                )

            sales_boost_hols = set(temp_df2[temp_df2['Jump from 7']>1.2]['description'])
            boost_map = { x : 1 if x in sales_boost_hols else  0 for x in temp_df2['description'].unique()}
            temp_df2['sales_boost_hol'] = temp_df2['description'].map(boost_map)

            train[family][store] = pd.merge(left = train[family][store],
                                            right = temp_df2.set_index('date')[['sales_boost_hol']],
                                            left_index = True,
                                            right_index = True,
                                            how = 'left')

            predictions[family][store] = pd.merge(left = predictions[family][store],
                                                right = temp_df2.set_index('date')[['sales_boost_hol']],
                                                left_index = True,
                                                right_index = True,
                                                how = 'left')

            train[family][store]['sales_boost_hol'].fillna(0, inplace=True)
            predictions[family][store]['sales_boost_hol'].fillna(0, inplace=True)
            train[family][store]['sales_boost_hol'] = train[family][store]['sales_boost_hol'].astype('int')
            predictions[family][store]['sales_boost_hol'] = predictions[family][store]['sales_boost_hol'].astype('int')

### Oil price

Since sales were higher with the oil price was lower, we add an oil price feature.

In [ ]:
def oil_prices():
    df['oil']['price_high'] = pd.cut(df['oil']['dcoilwtico'],bins=[0,75,999],labels=[0,1])
    df['oil']['price_high'] = df['oil']['price_high'].astype('int')
    for family in families:
        for store in train[family].keys():            
            train[family][store] = pd.merge(left = train[family][store],
                                            right = df['oil'][['price_high']],
                                            how='left',
                                            left_index = True,
                                           right_index = True)
            predictions[family][store] = pd.merge(left = predictions[family][store],
                                                  right = df['oil'][['price_high']],
                                                  how='left',
                                                  left_index = True,
                                                  right_index = True)

## Creating the features<a id='create'></a>

In [ ]:
def feature_engineering(nlags, load=False):
    if load:
        train_file1='../input/sales-forecasting-first-glance-eda/training_dic.pickle'
        pred_file1='../input/sales-forecasting-first-glance-eda/preds_dic.pickle'

        load_train(train_file1)
        load_pred(pred_file1)
    else:
        initialize_dfs()
        create_lags(nlags)
        seasonal_inds()
        holiday_sales()
        oil_prices()

In [ ]:
feature_engineering(nlags=400, load=True)

In [ ]:
# for family in families:
#     for store in train[family].keys():
#         for day in ['Tues','Weds','Thurs','Fri','Sat','Sun']:
#             train[family][store][day] = train[family][store][day].astype('int')
#             predictions[family][store][day] = predictions[family][store][day].astype('int')
#         train[family][store]['time_step'] = train[family][store]['time_step'].astype('int')
#         predictions[family][store]['time_step'] = predictions[family][store]['time_step'].astype('int')
#         train[family][store].drop_duplicates(inplace=True)
#         predictions[family][store].drop_duplicates(inplace=True)

In [ ]:
save_to_pickle(train, 'training_dic.pickle')
save_to_pickle(predictions, 'preds_dic.pickle')

# Model training<a id='models'></a>

We first fit a linear regression model to obtain a trend line. We then use XGBoost to predict the variation from the trend.

## Trend

For the families with logging issues, we start the trend training from June 2015. Below we plot the trend lines for beverages, which was a problem family.

[Return to top](#top)

In [ ]:
def trend_grid_search(load_models=False,param_grid_dict=None, verbose=0, return_models = True):
    '''
    Perform grid search for the trend part of the model.
    Fits best model found to each family/store and creates predictions for trend based on this.
    By default it performs a grid search.
    
    Args:
        load_models (bool; default False): if True then skips the grid search and loads pre-fitted models.
        
        param_grid_dict (dict; default None): Provide parameter grid for search.
                                              Configured as dict (keys = families) of dicts (keys = stores) of dicts (keys = parameters).
                                              If None then a default param_grid is supplied (size 3x3x3).
                                              
        verbose (int; default 0): 0 = no text output, 1 = partial output, 2 = full output.
    
        return_models (bool; default True): whether to return pre-fitted models or model information
    
    Returns:
        if return_models:
            dict: best models, pre-fitted
        elif load_models:
            returns nothing
        else:
            dict: best parameters (same configuration as param_grid_dict)
            dict: cv results (same configuration as param_grid_dict)
            DataFrame: best mean scores for each family/store pair        
    '''
    best_params = {} # dict to catch best parameters of each family/store pair
    cv_results = {}  # dict for cv results
    best_scores = {} # dict for mean of scores for best parameters
    if load_models:    
        with open('../input/sales-forecasting-first-glance-eda/trend_best_models.pickle', 'rb') as file:
            best_models = pickle.load(file)
    else:
        best_models = {}
    for family in families:
        t1=time()
        best_params[family] = {}
        cv_results[family] = {}
        best_scores[family] = {}
        if not load_models:
            best_models[family] = {}
        for store in train[family].keys():
            t0=time()
            pipe = Pipeline([('poly', PolynomialFeatures()),
                             ('reg', LinearRegression())#Ridge(max_iter = 30000, random_state=1, tol= 0.005))
                             ])
            if family in problem_families:
                mask = (train[family][store].index > '2015-06-01')
                X = train[family][store][mask][['time_step']]
                X_full = train[family][store][['time_step']]
                y = train[family][store][mask]['sales']
            else:
                X = train[family][store][['time_step']]
                X_full = X
                y = train[family][store]['sales']
    #         pipe.fit(X,y)

            # Either load the models or perform grid search
            if load_models:
                best_model=best_models[family][store]
            else:
                if isinstance(param_grid_dict, dict):
                    params = param_grid_dict[family][store]
                else:
                    params = {"poly__degree":[1,2,3]
#                           "elastic__alpha":[0.1,0.3,0.5,0.75,1],
#                           "elastic__l1_ratio":[0.1,0.3,0.5,0.7,0.9]
                         }
            
                nsplits = min(len(X) // 16, 6)  # ensures for recently opened stores/families there are not too many splits requested
                split = TimeSeriesSplit(n_splits=nsplits,test_size=16)
                grid_cv = GridSearchCV(pipe,
                                       params,
                                       cv=split,
                                       n_jobs=-1,
                                       return_train_score=True,
                                       refit=True, 
                                       scoring ='neg_mean_squared_error')
                grid_cv.fit(X,y)
                best_params[family][store]=grid_cv.best_params_
                cv_results[family][store] = grid_cv.cv_results_
                best_scores[family][store] = grid_cv.best_score_
                best_models[family][store] = grid_cv.best_estimator_
                best_model=grid_cv.best_estimator_

            # Get trend values
            train[family][store]['trend'] = best_model.predict(X_full)
            train[family][store]['residual'] = train[family][store]['sales'] - train[family][store]['trend']
            # Create predicted trend
            predictions[family][store]['trend'] = best_model.predict(predictions[family][store][['time_step']])

            # verbose
            if verbose > 1:
                print(f'{family} at store {store}, took {time()-t0:.2f} secs')
                print(f'Best params: {grid_cv.best_params_}')
                print(f'------------------------------------------------------------------------------------\
Mean score: {grid_cv.best_score_:.3f}')
        if verbose == 1:
            print(f'{family} took {time()-t1:.2f} secs')

    if verbose > 0:
        print('Finished grid search. Making scores dataframe.')
    best_scores_df = pd.DataFrame({ family : pd.Series(best_scores[family]) for family in families })
    if verbose > 0:
        print('Finished.')
    if return_models:
        return best_models
    elif load_models:
        return None
    else:
        return best_params, cv_results, best_scores_df

def fit_trend(load_models = True):
    if not load_models:
        best_models = trend_grid_search(load_models=False, param_grid_dict=None, verbose=1)
        save_to_pickle(best_models, 'trend_best_models.pickle')
    else:
        trend_grid_search(load_models=True)
        
    print('Trend predictions made')

In [ ]:
fit_trend(load_models = False)

## Cycle predictions

In [ ]:
def merge_from_generator(generator):
    '''
    The generator produces a pair of DFs at a time. 
    We merge all the first DFs in each pair, and all the second DFs in each pair into two DFs.
    '''
    get_df = generator
    dfA, dfB = next(get_df)
    next_dfA, next_dfB = next(get_df)
    n=2 # n counds the number of store DFs obtained
    while True:
        try:
            for df in [dfA,dfB,next_dfA, next_dfB]:
                df.drop_duplicates(inplace=True)   # without this we were getting duplacate rows, and the numbers grew exponentially with each merge, causing memory error
            # Merge store tables
            dfA = pd.merge(left = dfA, 
                                 right = next_dfA, 
                                 left_index=True, 
                                 right_index=True,
                                 how='outer')
            dfB = pd.merge(left = dfB, 
                                 right = next_dfB, 
                                 left_index=True, 
                                 right_index=True,
                                 how='outer')
            del next_dfA
            del next_dfB
            next_dfA, next_dfB = next(get_df)
            n+=1
        except StopIteration:
            break
    return dfA, dfB


def family_df():
    '''
    Generator that obtains family dataframes. Only the first has all the feature columns that are common to all.
    '''
    for family in families:
        t0=time()
        # Prepare the new dataframes - one for each family, joining the store tables
        def get_store_df(fam):
            for n, store in enumerate(train[fam].keys()):
                # Change name of sales and onpromotion store to indicate store
                sale_promo_change = {'sales' : f'{fam}_{store}_sales', 'onpromotion':f'{fam}_{store}_onpromotion', 'residual':f'{fam}_{store}_residual', 'trend':f'{fam}_{store}_trend'}
                tdf = train[fam][store].rename(columns = sale_promo_change)
                pdf = predictions[fam][store].rename(columns = sale_promo_change)
                # Change name of lag features to indicate store
                lag_cols = [ re.findall('sales_t-\d+', x)[0] for x in tdf.columns if not re.match('sales_t-', x)==None ]
                lag_change = { x : f'{fam}_{store}_{x}' for x in lag_cols }
                tdf.rename(columns = lag_change, inplace=True)
                pdf.rename(columns = lag_change, inplace=True)
                if n == 0 and fam == families[0]:
                    # The first DF keeps all columns
                    yield tdf, pdf
                else:
                    # Set the columns for the merge
                    store_cols = list(lag_change.values()) + list(sale_promo_change.values())
                    tdf = tdf[store_cols]
                    store_cols.remove(f'{family}_{store}_residual')
                    store_cols.remove(f'{family}_{store}_sales')
                    pdf = pdf[store_cols]
                    yield tdf, pdf

        train_df , pred_df = merge_from_generator(get_store_df(family))
        
        if family == families[0]:
            # Fill missing oil prices (at the weekends, use Friday's price)
            train_df['price_high'].fillna(method='ffill', inplace=True)
            pred_df['price_high'].fillna(method='ffill', inplace=True)

        # Fill missing lag sales / promos for stores that open after 2013-01-01
        NaN_cols =  [col for col in train_df.columns if train_df.isna().sum()[col]>0]
        train_df[NaN_cols] = train_df[NaN_cols].fillna(0)

        print(f'{family} took {time() - t0:.2f} secs to prep DFs')

        yield train_df, pred_df
    
def prep_df():
    mega_train, mega_pred = merge_from_generator(family_df())
    return mega_train, mega_pred
    
    
def get_training_sets(df_train, df_pred, pca=False, n_components=40):
    t0=time()
    # Set the columns for training
    family_store_pairs = []
    for family in families:
        for store in train[family].keys():
            family_store_pairs.append((family,store))
    train_features = list(df_train.columns)
    target_features = [f'{family}_{store}_residual' for family, store in family_store_pairs]
    drop_features1 = [f'{family}_{store}_sales' for family, store in family_store_pairs]
    drop_features2 = [f'{family}_{store}_trend' for family, store in family_store_pairs]
    drop_features = drop_features1 + drop_features2 + ['time_step']
    for col in drop_features + target_features:
        train_features.remove(col)
        
    # Get training data
    X = df_train[train_features]
    Y = df_train[target_features]
    X_forecast = df_pred[train_features]
    
    if pca:
        pca_pipe = Pipeline([('scaler',StandardScaler()),
                     ('pca',PCA(n_components = n_components))])
        pca_pipe.fit(X)
        for x in [X,X_forecast]:
            x = pd.DataFrame(pca_pipe.transform(x), index = x.index)
        exp_var = np.cumsum(pca_pipe['pca'].explained_variance_ratio_)[-1]
        
        print(f'PCA uses {exp_var:.2f} of variance')
        
        
    print(f'It took {time() - t0:.2f} secs to get training sets')
    
    return X, Y, X_forecast



def xgb_fit_predict(X, Y, X_forecast, verbose = 0, n_iter=70):
    t0=time()
    # Fit the model
    xgbforgrid = MultiOutputRegressor(XGBRegressor())
    split = TimeSeriesSplit(n_splits=min(2, len(X)//16), test_size = 16)
    
      
    gridsearch = RandomizedSearchCV(xgbforgrid,cv=split, n_iter=n_iter,
                                    param_distributions={'estimator__colsample_bytree' : uniform(loc=0.2, scale=0.8),
                                                       'estimator__learning_rate' : beta(a=4, b=30),
                                                       'estimator__n_estimators' : nbinom(10,0.07,loc=1),
                                                       'estimator__max_depth' : nbinom(6,0.6,loc=1)
                                                             }, n_jobs=-1,verbose=verbose)
    
    gridsearch.fit(X,Y) 
    xgb=gridsearch.best_estimator_
    with open('xgb_cv.pickle','wb') as file:
        pickle.dump(gridsearch.cv_results_, file, protocol=-1)
    xgb.fit(X,Y)
    # Obtain predictions
    Y_pred = pd.DataFrame(xgb.predict(X), columns = [f'{x}_pred' for x in Y.columns], index=X.index)
    Y_forecast = pd.DataFrame(xgb.predict(X_forecast), columns = Y.columns, index=X_forecast.index)
    
    # Merge predictions into DFs
    df_train_out = pd.merge(left = fdf_train,
                                    right = Y_pred,
                                    left_index=True,
                                    right_index=True)
    df_pred_out  = pd.merge(left = df_pred,
                                    right = Y_forecast,
                                    left_index=True,
                                    right_index=True)
    if verbose>0:
        display(family_pred[family])
    
    print(f'{family} took {time() - t0:.2f} secs to fit XGB and predict values')
    
    return df_train_out, df_pred_out

In [ ]:
def cycle_model():
    tdf, pdf = prep_df()
    imp = SimpleImputer(strategy='median')
    tdf_imp = pd.DataFrame(imp.fit_transform(tdf), columns = tdf.columns, index = tdf.index)
    X, Y, X_forecast = get_training_sets(tdf_imp, pdf, pca=True, n_components = 10)
    tdf_out, pdf_out = xgb_fit_predict(X,Y,X_forecast, n_iter=2, verbose = 3)
    return tdf_out, pdf_out

In [ ]:
tdf_out, pdf_out = cycle_model()
with open('tdf_with_preds.pickle','wb') as file:
        pickle.dump(tdf_out, file, protocol=-1)
with open('pdf_with_preds.pickle','wb') as file:
        pickle.dump(pdf_out, file, protocol=-1)

In [ ]:

# for obj, name in zip([family_train, family_pred, train, predictions],
#                      ['family_train', 'family_pred', 'train', 'predictions']):
#     with open(f'{name}.pickle', 'wb') as file:
#         pickle.dump(obj, file, protocol=-1)

# # train = load_pickle('train')
# # predictions = load_pickle('predictions')

In [ ]:
# for family in families:
#     for store in train[family].keys():
#         preds = (family_train[family][f'{store}_residual_pred'].copy() + train[family][store]['trend'].copy()).apply(lambda x : x if x > 0 else 0).to_frame()
#         train[family][store] = pd.merge(left = train[family][store], right = preds, left_index = True, right_index = True).rename(columns={0:'pred'})
#         predictions[family][store]['sales'] = (family_pred[family][f'{store}_residual'] + predictions[family][store]['trend']).apply(lambda x : x if x > 0 else 0)

In [ ]:
# fig,ax=plt.subplots(10,4,figsize = (25,20), sharex=True)
# for j,family in enumerate(['GROCERY I', 'BEVERAGES','FROZEN FOODS', 'SCHOOL AND OFFICE SUPPLIES']):
#     for i,store in enumerate(range(28,38)):
#         for col, color in zip(['sales','pred','trend'],['black','orange','blue']):
#             train[family][store].reset_index()[(train[family][store].index.year == 2017) 
#                                                & (train[family][store].index.month >=6)]\
#             .plot(x='date' , y=col, ax=ax[i,j],lw=0.7,alpha=0.7, c=color)
# #         predictions[family][store].reset_index()\
# #             .plot(x='date' , y='sales', ax=ax[i,j],lw=0.7,alpha=0.7, c='red', label = 'prediction')
#         ax[i,j].legend(loc='lower left')
# fig.suptitle('Forecasted sales for some families in stores 1 - 10')
# fig.tight_layout()
# fig.subplots_adjust(top=0.96)
# plt.show()

# Evaluation<a id='eval'></a>

The heatmap below shows the root mean squared log error on the training set for each product family and store.

[Return to top](#top)

In [ ]:
# rmsle = {}
# for family in families:
#     rmsle[family] = {}
#     for store in train[family].keys():
#         if family in problem_families:
#             mask = (train[family][store].index > '2015-06-01')
#             y=train[family][store][mask]['sales']
#             y_pred=train[family][store][mask]['pred']
#         else:
#             y=train[family][store]['sales']
#             y_pred=train[family][store]['pred']
#         rmsle[family][store] = np.sqrt(msle(y, y_pred))
        
# rmsle_df = pd.DataFrame({ family : pd.Series(rmsle[family]) for family in families })

# fig,ax=plt.subplots(figsize=(20,16))
# sns.heatmap(data = rmsle_df,
#             annot=True,
#             fmt=".2f",
#             linewidths=0.1,
#             ax=ax,
#             vmax = 1,
#             vmin = 0,
#             annot_kws = {'fontsize':8},
#             cmap = 'cool'
#            )
# plt.show()

In [ ]:
# weak_families = [ family for family in families if rmsle_df[family].mean() > 0.15 ]
# weak_stores = [store for store in stores if rmsle_df.loc[store].mean() > 0.1]

# print(f'The weak families are: {weak_families}')
# print(f'The weak stores are: {weak_stores}')

# # if len(weak_families) >0 and len(weak_stores)>0:

# #     fig,ax = plt.subplots(len(weak_stores), len(weak_families), figsize=(45,20),sharex=True)
# #     for j,family in enumerate(weak_families):
# #         for i,store in enumerate(weak_stores):
# #             if store in train[family].keys():
# #                 residual = train[family][store]['sales'] - train[family][store]['pred']
# #                 residual_rolling = residual.rolling(window=16).mean()
# #                 residual_rolling.plot(ax=ax[i,j])
# #                 ax[i,j].set_title(f'{family} at store {store}; rmsle={rmsle[family][store]:.2f}')
# #                 ax[i,j].hlines(0,pd.Timestamp('2013-01-01'),pd.Timestamp('2017-08-15'), colors = 'black')
# #             fig.subplots_adjust(wspace=0.1, hspace=0.3)
# #     plt.show()

## Create submission

In [ ]:
# submission = pd.DataFrame(columns = ['id','sales'])
# for family in families:
#     for store in stores:
#         mask = (df['test']['store_nbr'] == store) & (df['test']['family'] == family)
#         predictions[family][store]['id'] = list(df['test'][mask].reset_index()['id'])
#         submission = pd.concat([submission, predictions[family][store][['id','sales']]])
        
# submission.sort_values('id', inplace=True)
# submission.to_csv('submission.csv', index=False)